# 쇼핑몰 단골 찾기: 고객 분석 보고서

쇼핑몰 거래 기록을 정리·분석해 **"누가 최근에, 자주, 많이 사는가"** 로 고객을 나누고, 그룹별 마케팅 전략을 제안하는 분석 보고서입니다.

| 데이터 | 출처 | 라이선스 | 쓰임 |
|---|---|---|---|
| Online Retail II (거래 기록) | UCI Machine Learning Repository, Daqing Chen (DOI 10.24432/C5CG6D) | CC BY 4.0 | 결측치·이상치·통계·RFM |
| Fashion Product Images (Small) | Kaggle, Param Aggarwal | MIT (Kaggle 페이지 표기) | 상품 정보·이미지 피처 |

> 두 데이터는 서로 연결되지 않습니다(같은 상품이 아님). 그래서 **거래 분석과 이미지 피처 실습은 별도 데이터로** 진행합니다.

**목차** (단계가 진행되면서 채워집니다)
1. 데이터 개요 ← 현재
2. 정제와 결측치 처리
3. 이상치 탐지와 처리
4. 텍스트·이미지 피처
5. 통계 분석과 시각화
6. RFM 고객 세분화

## 0. 준비

사용할 도구를 불러오고 데이터 폴더 위치를 정합니다.

In [1]:
import pandas as pd            # 표 데이터를 다루는 라이브러리 (pd 라는 짧은 이름으로 부름)
from pathlib import Path       # 파일 경로를 다루는 파이썬 기본 도구

DATA_DIR = Path("..") / "data"             # 이 노트북(notebooks/) 기준으로 한 칸 위의 data 폴더
pd.set_option("display.max_columns", 20)   # 컬럼이 많아도 ... 로 줄이지 않고 모두 보여주기
pd.set_option("display.width", 200)        # 한 줄에 보여줄 글자 폭

## 1. 거래 데이터: Online Retail II

영국 온라인 선물용품 쇼핑몰의 2009-12 ~ 2011-12 거래 기록입니다. 이 데이터로 고객을 분석합니다.

### 1-0. 불러오기

CSV를 읽을 때 두 가지를 미리 알려줍니다.
- `Invoice`(주문번호)와 `StockCode`(상품코드)는 `C489449`처럼 **글자가 섞인 값**이 있어서, 숫자가 아니라 글자(`str`)로 읽습니다.
- `InvoiceDate`는 글자가 아니라 **날짜 형식**으로 읽어야 기간 계산을 할 수 있습니다(`parse_dates`).

In [2]:
retail = pd.read_csv(
    DATA_DIR / "online_retail_II.csv",
    dtype={"Invoice": "str", "StockCode": "str"},   # 'C'로 시작하는 주문번호가 있어 글자로 읽음
    parse_dates=["InvoiceDate"],                    # 날짜 글자를 진짜 날짜로 변환
)
print(f"행 {retail.shape[0]:,}개 × 컬럼 {retail.shape[1]}개")

행 1,067,371개 × 컬럼 8개


### 1-1. `head()`: 맨 위 5줄

데이터가 어떤 모양인지 눈으로 먼저 확인합니다.

In [3]:
retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


> **해석**
>
> **수치 근거:** 맨 위 5줄이 모두 주문번호 `489434`, 고객 `13085`, 시각 `2009-12-01 07:45`로 같고, 상품(`StockCode`)만 서로 다릅니다.
>
> **시사점:** 한 줄은 "한 주문 안의 상품 한 종류"입니다. 그래서 고객이 몇 번 샀는지(Frequency)는 줄 수가 아니라 **고유 주문번호의 개수**로 세어야 합니다.

### 1-2. `info()`: 구조와 결측 확인

컬럼별로 값이 몇 개 들어 있는지(`Non-Null Count`)와 자료형(`Dtype`)을 보여줍니다. 전체 행 수보다 적으면 그만큼 **비어 있다(결측)** 는 뜻입니다.

In [4]:
retail.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 65.1 MB


> **해석**
>
> **수치 근거:** 전체 1,067,371행 × 8컬럼이며, `Customer ID`는 824,364행에만 값이 있어 **243,007행(22.77%)이 비어 있습니다.** `Description`(상품명)은 4,382행(0.41%)이 비어 있습니다. 자료형은 글자 4개, 소수 2개, 정수 1개, 날짜 1개입니다.
>
> **시사점:** 수치·범주·날짜·텍스트가 섞여 있어 "3가지 이상 유형" 조건을 충족합니다. 고객 ID가 없는 행은 "누가 샀는지" 알 수 없어 고객 분석(RFM)에서 제외해야 하고(전체의 약 1/4에 가까움), 상품명 결측은 0.41%로 적어 정제 뒤에도 남는지 점검합니다(2장).
>
> (`Customer ID`가 정수가 아니라 소수형 `float64`인 이유: 값이 비어 있는 행(`NaN`)을 담으려면 소수형이어야 하기 때문입니다.)

### 1-3. `describe()`: 숫자 요약

숫자 컬럼의 개수(`count`), 평균(`mean`), 표준편차(`std`), 최솟값(`min`), 25%·50%(중앙값)·75% 지점, 최댓값(`max`)을 한 번에 보여줍니다. `Customer ID`는 식별 번호라 평균이 의미 없어서 뺐습니다.

In [5]:
retail[["Quantity", "Price"]].describe().style.format("{:,.2f}")   # 천 단위 쉼표, 소수 둘째 자리까지

,Quantity,Price
count,"1,067,371.00","1,067,371.00"
mean,9.94,4.65
std,172.71,123.55
min,"-80,995.00","-53,594.36"
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,"80,995.00","38,970.00"


describe의 숫자가 왜 그런지 확인하려고, 극단값과 음수 행을 직접 들여다봅니다.

In [6]:
is_cancel = retail["Invoice"].str.startswith("C")      # 주문번호가 'C'로 시작 = 취소 주문
neg_qty = retail["Quantity"] < 0                        # 수량이 음수인 행

print(f"수량이 음수인 행:        {neg_qty.sum():>7,}개 ({neg_qty.mean():.2%})")
print(f"  └ 그중 취소(C) 주문:   {(neg_qty & is_cancel).sum():>7,}개")
print(f"  └ 그중 취소 표시 없음: {(neg_qty & ~is_cancel).sum():>7,}개")
print(f"가격이 0인 행:           {(retail['Price'] == 0).sum():>7,}개 ({(retail['Price'] == 0).mean():.2%})")
print(f"가격이 음수인 행:        {(retail['Price'] < 0).sum():>7,}개")
print(f"거래 기간: {retail['InvoiceDate'].min()} ~ {retail['InvoiceDate'].max()}")
print()
print("▶ 수량이 가장 작은 행 / 가장 큰 행")
print(retail.loc[[retail["Quantity"].idxmin(), retail["Quantity"].idxmax()],
                 ["Invoice", "Description", "Quantity", "Price", "Customer ID"]])
print()
print("▶ 가격이 가장 작은 행")
print(retail.loc[[retail["Price"].idxmin()], ["Invoice", "Description", "Quantity", "Price"]])

수량이 음수인 행:         22,950개 (2.15%)
  └ 그중 취소(C) 주문:    19,493개
  └ 그중 취소 표시 없음:   3,457개
가격이 0인 행:             6,202개 (0.58%)
가격이 음수인 행:              5개
거래 기간: 2009-12-01 07:45:00 ~ 2011-12-09 12:50:00

▶ 수량이 가장 작은 행 / 가장 큰 행
         Invoice                  Description  Quantity  Price  Customer ID
1065883  C581484  PAPER CRAFT , LITTLE BIRDIE    -80995   2.08      16446.0
1065882   581483  PAPER CRAFT , LITTLE BIRDIE     80995   2.08      16446.0

▶ 가격이 가장 작은 행
        Invoice      Description  Quantity     Price
179403  A506401  Adjust bad debt         1 -53594.36


> **해석**
>
> **수치 근거:** `Quantity`는 중앙값 3, 평균 9.94, 75% 지점 10인데 최댓값은 80,995, 최솟값은 −80,995입니다. `Price`도 중앙값 2.10, 평균 4.65인데 최솟값 −53,594.36, 최댓값 38,970입니다. 수량이 음수인 행은 22,950개(2.15%)이고 그중 19,493개가 주문번호가 `C`로 시작하는 취소 주문입니다. 가격이 0인 행도 6,202개(0.58%) 있습니다.
>
> **시사점:** `Quantity`는 평균(9.94)이 중앙값(3)의 3배가 넘어 **오른쪽으로 길게 치우친 분포**이고, 극단값이 평균과 표준편차를 끌고 갑니다. 그래서 이상치 판단에는 극단값에 덜 흔들리는 **사분위수 기반 IQR**이 적합합니다(3장). 또 취소·음수·0 이하 행은 매출 계산을 왜곡하므로 정제 단계에서 제거합니다(2장).
>
> **눈여겨볼 두 행:**
> - 수량의 최댓값 80,995와 최솟값 −80,995는 **같은 상품·같은 고객(16446)이 대량 주문(`581483`)했다가 곧바로 취소(`C581484`)한 쌍**입니다.
> - 가격의 최솟값 −53,594.36은 상품명이 `Adjust bad debt`(부실채권 조정)인 **회계 조정 행**이라 실제 판매가 아닙니다.

## 2. 상품 데이터: Fashion Product Images

패션 상품 약 4.4만 개의 정보 표(`styles.csv`)입니다. 이 표의 `id`와 같은 이름의 사진 파일(`images/{id}.jpg`)이 있어, 나중에 사진을 숫자로 바꾸는 실습(4장)에 씁니다.

### 2-0. 불러오기

`styles.csv`는 일부 상품명에 쉼표(`,`)가 들어 있는데 따옴표로 감싸지 않아, 컬럼 수가 어긋나는 줄이 **22줄** 있습니다. 이 줄들은 건너뛰고(`on_bad_lines="skip"`) 읽으며, 전체의 0.05%라 분석에는 영향이 거의 없습니다.

In [7]:
styles = pd.read_csv(
    DATA_DIR / "fashion_small" / "styles.csv",
    on_bad_lines="skip",          # 컬럼 수가 어긋난 22줄은 건너뜀
)
print(f"행 {styles.shape[0]:,}개 × 컬럼 {styles.shape[1]}개")

행 44,424개 × 컬럼 10개


### 2-1. `head()`

In [8]:
styles.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


> **해석**
>
> **수치 근거:** 한 줄이 상품 한 개입니다(예: `id` 15970은 남성 네이비 셔츠). 컬럼은 `id`, `gender`, `masterCategory`, `subCategory`, `articleType`, `baseColour`, `season`, `year`, `usage`, `productDisplayName` 10개입니다.
>
> **시사점:** `id`가 `images/15970.jpg`처럼 사진 파일 이름과 같아서, 이 표와 이미지를 `id`로 연결할 수 있습니다(4장). 상품명(`productDisplayName`)은 글자 수·단어 수를 센 파생 변수의 재료가 됩니다.

### 2-2. `info()`

In [9]:
styles.info()

<class 'pandas.DataFrame'>
RangeIndex: 44424 entries, 0 to 44423
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  44424 non-null  int64  
 1   gender              44424 non-null  str    
 2   masterCategory      44424 non-null  str    
 3   subCategory         44424 non-null  str    
 4   articleType         44424 non-null  str    
 5   baseColour          44409 non-null  str    
 6   season              44403 non-null  str    
 7   year                44423 non-null  float64
 8   usage               44107 non-null  str    
 9   productDisplayName  44417 non-null  str    
dtypes: float64(1), int64(1), str(8)
memory usage: 3.4 MB


> **해석**
>
> **수치 근거:** 44,424행 × 10컬럼이고, 결측은 `usage` 317개(0.71%), `season` 21개, `baseColour` 15개, `productDisplayName` 7개, `year` 1개로 **모두 1% 미만**입니다.
>
> **시사점:** 결측이 적어 지워도 분석에 큰 영향은 없지만, 오히려 "그룹별 대치"를 보여주기에 알맞은 대상입니다. 같은 품목(`articleType`)끼리 묶어 가장 흔한 값으로 채우는 방식을 2장에서 적용합니다.

### 2-3. `describe()`와 분포 확인

숫자 컬럼은 `year` 하나뿐이고(`id`는 식별 번호라 평균이 의미 없음), 나머지는 글자(범주)라서 값별 개수(`value_counts`)로 분포를 봅니다.

In [10]:
print(styles[["year"]].describe().round(1))
print()
print("▶ year 상위 3개\n", styles["year"].value_counts().head(3), sep="")
print()
print("▶ gender\n", styles["gender"].value_counts(), sep="")
print()
print("▶ masterCategory 상위 3개\n", styles["masterCategory"].value_counts().head(3), sep="")

          year
count  44423.0
mean    2012.8
std        2.1
min     2007.0
25%     2011.0
50%     2012.0
75%     2015.0
max     2019.0

▶ year 상위 3개
year
2012.0    16288
2011.0    13672
2016.0     6006
Name: count, dtype: int64

▶ gender
gender
Men       22147
Women     18631
Unisex     2161
Boys        830
Girls       655
Name: count, dtype: int64

▶ masterCategory 상위 3개
masterCategory
Apparel        21397
Accessories    11274
Footwear        9219
Name: count, dtype: int64


> **해석**
>
> **수치 근거:** `year`는 2007~2019년이고 중앙값은 2012년이며, 2012년(16,288개)과 2011년(13,672개)이 전체의 약 67%를 차지합니다. 성별은 Men 22,147개(49.9%), Women 18,631개(41.9%)이고, 대분류는 Apparel(의류) 21,397개(48.2%)로 가장 많습니다.
>
> **시사점:** 특정 연도·성별·의류에 치우친 표본이므로, 이후 이미지 평균·표준편차 같은 결과를 말할 때 "의류 중심 상품 사진"이라는 한계를 함께 밝혀야 합니다.

## 3. 1장에서 알게 된 것과 다음 단계

| 발견 | 근거 | 다음 단계에서 할 일 |
|---|---|---|
| 한 주문에 여러 줄이 있음 | 맨 위 5줄이 같은 주문번호 | 구매 횟수(F)는 고유 주문번호 수로 계산 (6장) |
| 고객 ID가 22.77% 비어 있음 | `info()` | RFM 대상에서 제외 (2장, 6장) |
| 취소·음수·0 가격 행 | 수량 음수 22,950개, 가격 0 6,202개 | 정제로 제거 (2장) |
| 극단값이 평균을 끌고 감 | 수량 ±80,995 | IQR로 이상치 탐지 (3장) |
| 두 시트가 같은 기간을 중복 포함 | 2010-12-01~09의 22,523행씩 겹침 (원본 점검에서 확인) | 중복 행 제거 후 개수 기록 (2장) |
| 상품 표에 결측이 있음 | `usage` 317개 등 | 그룹별 대치 (2장) |
| `styles.csv` 22줄의 형식 오류 | 상품명 속 쉼표 | 건너뛰고 개수를 README에 기록 |